In [ ]:
# Cell 1: Codex suggested first cell

import torch, whisper

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FP16 = DEVICE == "cuda"
OUT_DIR = "transcription_outputs"

print("Device:", DEVICE)


Device: cuda


In [7]:
# Cell 2: Imports and configuration

from __future__ import annotations

import os
import shutil
from pathlib import Path
from pprint import pprint
from typing import Any


# ---- User-configurable paths ----
AUDIO_PATH = Path("doctor_patient_example.wav")   # change this to your file
OUTPUT_DIR = Path("transcription_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TRANSCRIPT_TXT_PATH = OUTPUT_DIR / f"{AUDIO_PATH.stem}_transcript.txt"


def preview_dict(d: dict[str, Any], max_items: int = 8) -> dict[str, Any]:
    """
    Return a shallow preview of a dictionary for readable printing.
    Useful when Whisper returns nested objects and you want a quick look.
    """
    out = {}
    for i, (k, v) in enumerate(d.items()):
        if i >= max_items:
            break
        out[k] = v
    return out


print("Audio path:", AUDIO_PATH)
print("Output directory:", OUTPUT_DIR.resolve())
print("Transcript txt path:", TRANSCRIPT_TXT_PATH.resolve())

Audio path: doctor_patient_example.wav
Output directory: /home/deuterium/whisper_app/transcription_outputs
Transcript txt path: /home/deuterium/whisper_app/transcription_outputs/doctor_patient_example_transcript.txt


In [8]:
# Cell 3: Validate input path

def validate_audio_input(audio_path: Path) -> None:
    """
    Basic validation before attempting transcription.
    This keeps failures obvious and early.
    """
    if not audio_path.exists():
        raise FileNotFoundError(
            f"Audio file not found: {audio_path}\n"
            "Update AUDIO_PATH to point to a real audio file."
        )

    if not audio_path.is_file():
        raise ValueError(f"Path is not a file: {audio_path}")

    if audio_path.stat().st_size == 0:
        raise ValueError(f"Audio file is empty: {audio_path}")

    if shutil.which("ffmpeg") is None:
        raise EnvironmentError(
            "ffmpeg was not found on your system PATH.\n"
            "Install ffmpeg first, then rerun the notebook.\n"
            "On Ubuntu, that is commonly: sudo apt-get install ffmpeg"
        )


validate_audio_input(AUDIO_PATH)
print("Input validation passed.")

Input validation passed.


In [9]:
# Cell 4: Load Whisper model

import whisper


# Keep the model small for a toy notebook.
# You can later try: "base", "small", "medium", "large"
MODEL_NAME = "base"

print(f"Loading Whisper model: {MODEL_NAME}")
model = whisper.load_model(MODEL_NAME)
print("Model loaded successfully.")

Loading Whisper model: base


100%|███████████████████████████████████████| 139M/139M [00:04<00:00, 29.5MiB/s]


Model loaded successfully.


In [10]:
# Cell 5: Run transcription

def transcribe_audio(model: Any, audio_path: Path) -> dict[str, Any]:
    """
    Run Whisper transcription and return the near-raw Whisper result dict.

    We keep the options simple and explicit so you can see
    what is being passed into the model.
    """
    result = model.transcribe(
        str(audio_path),
        task="transcribe",
        fp16=False,          # safer default on CPU; Whisper falls back to FP32 on CPU. :contentReference[oaicite:2]{index=2}
        verbose=False,
        word_timestamps=False
    )
    return result


whisper_result = transcribe_audio(model, AUDIO_PATH)

print("Transcription complete.")
print("Top-level text preview:")
print(whisper_result["text"][:500])

Detected language: English


100%|██████████| 5002/5002 [00:02<00:00, 2311.10frames/s]

Transcription complete.
Top-level text preview:
 Okay, I'm the doctor. Good morning. Can you tell me what brings you in today? Oh, yeah, I've been refilling really thoroughly, like even after sleeping out at Cupic's Austin. I see. Have you noticed any other symptoms like headaches, dizziness or shortness of breath? Sometimes I get this weird test tightness, but I'm not sure if it's the stress or something else. Okay, that's important. Do you smoke? Drink, or exercise regularly? I don't smoke, but I do drink one weekend and honestly I have an 


In [11]:
# Cell 6: Inspect Whisper output structure

print("Type returned by model.transcribe(...):")
print(type(whisper_result))

print("\nTop-level keys:")
print(list(whisper_result.keys()))

print("\nTop-level preview:")
pprint(preview_dict(whisper_result))

segments = whisper_result.get("segments", [])
print("\nNumber of segments:")
print(len(segments))

if segments:
    first_segment = segments[0]
    print("\nType of first segment:")
    print(type(first_segment))

    print("\nKeys in first segment:")
    print(list(first_segment.keys()))

    print("\nFirst segment preview:")
    pprint(first_segment)

    print("\nReadable sample fields from first segment:")
    print("id:", first_segment.get("id"))
    print("start:", first_segment.get("start"))
    print("end:", first_segment.get("end"))
    print("text:", repr(first_segment.get("text")))
    print("avg_logprob:", first_segment.get("avg_logprob"))
    print("no_speech_prob:", first_segment.get("no_speech_prob"))
    print("compression_ratio:", first_segment.get("compression_ratio"))
else:
    print("No segments were returned.")

Type returned by model.transcribe(...):
<class 'dict'>

Top-level keys:
['text', 'segments', 'language']

Top-level preview:
{'language': 'en',
 'segments': [{'avg_logprob': -0.33188512941069953,
               'compression_ratio': 1.579268292682927,
               'end': 4.8,
               'id': 0,
               'no_speech_prob': 0.24799808859825134,
               'seek': 0,
               'start': 0.0,
               'temperature': 0.0,
               'text': " Okay, I'm the doctor.",
               'tokens': [50364, 1033, 11, 286, 478, 264, 4631, 13, 50604]},
              {'avg_logprob': -0.33188512941069953,
               'compression_ratio': 1.579268292682927,
               'end': 5.8,
               'id': 1,
               'no_speech_prob': 0.24799808859825134,
               'seek': 0,
               'start': 4.8,
               'temperature': 0.0,
               'text': ' Good morning.',
               'tokens': [50604, 2205, 2446, 13, 50654]},
              {'avg_logprob

In [12]:
# Cell 7: Create intermediate pipeline schema

def build_transcript_segments(whisper_result: dict[str, Any]) -> list[dict[str, Any]]:
    """
    Standardize Whisper's segment output into a cleaner intermediate schema.

    This is the object you can hand to later pipeline stages such as:
    - diarization alignment
    - chunk merging
    - summarization
    - note generation
    - classification
    """
    transcript_segments: list[dict[str, Any]] = []

    for seg in whisper_result.get("segments", []):
        transcript_segments.append(
            {
                "segment_id": seg.get("id"),
                "start_sec": seg.get("start"),
                "end_sec": seg.get("end"),
                "duration_sec": (
                    seg.get("end") - seg.get("start")
                    if seg.get("start") is not None and seg.get("end") is not None
                    else None
                ),
                "text": (seg.get("text") or "").strip(),
                "tokens": seg.get("tokens"),  # token ids from Whisper, useful for debugging/advanced inspection
                "avg_logprob": seg.get("avg_logprob"),  # heuristic quality signal, not calibrated confidence
                "no_speech_prob": seg.get("no_speech_prob"),  # useful when filtering silence-like regions
                "compression_ratio": seg.get("compression_ratio"),  # useful for detecting odd decoding behavior
                "temperature": seg.get("temperature"),
                "source": "whisper"
            }
        )

    return transcript_segments


transcript_segments = build_transcript_segments(whisper_result)

# Higher-level object you can hand off to later stages
transcription_result = {
    "audio_path": str(AUDIO_PATH),
    "language": whisper_result.get("language"),
    "full_text": whisper_result.get("text", "").strip(),
    "transcript_segments": transcript_segments,
    "segment_count": len(transcript_segments),
    "engine": "openai-whisper",
    "model_name": MODEL_NAME,
}

pipeline_ready_transcript = transcription_result

print("Built intermediate schema: transcript_segments")
print("Number of standardized segments:", len(transcript_segments))

if transcript_segments:
    print("\nFirst standardized segment:")
    pprint(transcript_segments[0])

print("\nPipeline-ready top-level object keys:")
print(list(pipeline_ready_transcript.keys()))

Built intermediate schema: transcript_segments
Number of standardized segments: 22

First standardized segment:
{'avg_logprob': -0.33188512941069953,
 'compression_ratio': 1.579268292682927,
 'duration_sec': 4.8,
 'end_sec': 4.8,
 'no_speech_prob': 0.24799808859825134,
 'segment_id': 0,
 'source': 'whisper',
 'start_sec': 0.0,
 'temperature': 0.0,
 'text': "Okay, I'm the doctor.",
 'tokens': [50364, 1033, 11, 286, 478, 264, 4631, 13, 50604]}

Pipeline-ready top-level object keys:
['audio_path', 'language', 'full_text', 'transcript_segments', 'segment_count', 'engine', 'model_name']


In [13]:
# Cell 8: Print the most pipeline-relevant fields

full_transcript_text = whisper_result.get("text", "").strip()

segment_level_text = [seg["text"] for seg in transcript_segments]
segment_times = [
    {
        "segment_id": seg["segment_id"],
        "start_sec": seg["start_sec"],
        "end_sec": seg["end_sec"]
    }
    for seg in transcript_segments
]
segment_quality_fields = [
    {
        "segment_id": seg["segment_id"],
        "avg_logprob": seg["avg_logprob"],
        "no_speech_prob": seg["no_speech_prob"],
        "compression_ratio": seg["compression_ratio"],
        "temperature": seg["temperature"]
    }
    for seg in transcript_segments
]

print("1. Full transcript text:")
print(full_transcript_text[:1000])

print("\n2. First 3 segment-level texts:")
for item in segment_level_text[:3]:
    print("-", item)

print("\n3. First 3 segment start/end times:")
for item in segment_times[:3]:
    print(item)

print("\n4. First 3 segment metadata / quality-related fields:")
for item in segment_quality_fields[:3]:
    print(item)

1. Full transcript text:
Okay, I'm the doctor. Good morning. Can you tell me what brings you in today? Oh, yeah, I've been refilling really thoroughly, like even after sleeping out at Cupic's Austin. I see. Have you noticed any other symptoms like headaches, dizziness or shortness of breath? Sometimes I get this weird test tightness, but I'm not sure if it's the stress or something else. Okay, that's important. Do you smoke? Drink, or exercise regularly? I don't smoke, but I do drink one weekend and honestly I have an exercise in months. Thanks for being honest. Are you currently taking medications or supplements? Just some other counter vitamins. Alright, cool. From what? You shared it could be life's sorry related, but I'll turn some blood tests. Sure, don't worry. Well, forget this out together. I'm just turning.

2. First 3 segment-level texts:
- Okay, I'm the doctor.
- Good morning.
- Can you tell me what brings you in today?

3. First 3 segment start/end times:
{'segment_id': 0, 

In [14]:
# Cell 9: Save transcript

def save_plain_transcript(text: str, output_path: Path) -> None:
    """
    Save plain transcript text to disk.
    """
    output_path.write_text(text, encoding="utf-8")


save_plain_transcript(full_transcript_text, TRANSCRIPT_TXT_PATH)

print(f"Saved plain transcript to: {TRANSCRIPT_TXT_PATH.resolve()}")

Saved plain transcript to: /home/deuterium/whisper_app/transcription_outputs/doctor_patient_example_transcript.txt


In [15]:
# Cell 10: Optional sanity preview of the final handoff object

print("Pipeline-ready transcript summary:")
print({
    "audio_path": pipeline_ready_transcript["audio_path"],
    "language": pipeline_ready_transcript["language"],
    "segment_count": pipeline_ready_transcript["segment_count"],
    "engine": pipeline_ready_transcript["engine"],
    "model_name": pipeline_ready_transcript["model_name"],
})

print("\nFirst 2 standardized transcript segments:")
pprint(pipeline_ready_transcript["transcript_segments"][:2])

Pipeline-ready transcript summary:
{'audio_path': 'doctor_patient_example.wav', 'language': 'en', 'segment_count': 22, 'engine': 'openai-whisper', 'model_name': 'base'}

First 2 standardized transcript segments:
[{'avg_logprob': -0.33188512941069953,
  'compression_ratio': 1.579268292682927,
  'duration_sec': 4.8,
  'end_sec': 4.8,
  'no_speech_prob': 0.24799808859825134,
  'segment_id': 0,
  'source': 'whisper',
  'start_sec': 0.0,
  'temperature': 0.0,
  'text': "Okay, I'm the doctor.",
  'tokens': [50364, 1033, 11, 286, 478, 264, 4631, 13, 50604]},
 {'avg_logprob': -0.33188512941069953,
  'compression_ratio': 1.579268292682927,
  'duration_sec': 1.0,
  'end_sec': 5.8,
  'no_speech_prob': 0.24799808859825134,
  'segment_id': 1,
  'source': 'whisper',
  'start_sec': 4.8,
  'temperature': 0.0,
  'text': 'Good morning.',
  'tokens': [50604, 2205, 2446, 13, 50654]}]
